# Ü4.1 — DynamicFrames im Notebook erkunden

Interaktive Glue-Session. Ablauf: `raw.orders` lesen → ApplyMapping → Filter →
`toDF()` + Spark SQL → Parquet schreiben → zählen.

**Voraussetzung:** Crawler aus Ü3.1 hat `raw.orders` katalogisiert.
Ausgabe landet unter `s3://gfu-glue-training-629452195361/processed/orders_nb/`
(eigener Prefix, kollidiert nicht mit dem Job-Output aus Ü5.1).

Die Code-Zellen sind **vorgefertigt** — die Syntax steht schon. Du ersetzt nur
die `____name`-Blanks und die `<...>`-Platzhalter. Über jeder Zelle steht als
Kommentar eine Leitfrage pro Blank.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ApplyMapping, Filter
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

## 1) Lesen — DynamicFrame aus dem Katalog

In [ ]:
# Fülle die ____ aus. Syntax steht — du entscheidest nur, WAS wohin gehört.
#   ____datenbank : in welcher Katalog-Datenbank liegt die Tabelle?
#   ____tabelle   : welche Tabelle willst du lesen?
orders = glueContext.create_dynamic_frame.from_catalog(
    database="____datenbank", table_name="____tabelle", transformation_ctx="orders"
)
orders.printSchema()
orders.toDF().show(truncate=False)

## 2) ApplyMapping — Spalten bereinigen

Schau ins Schema von oben: manche Rohspalten haben Leerzeichen im Namen oder den
falschen Typ. Jedes Tupel sagt: `("<quellspalte>", "<quelltyp>", "<zielspalte>", "<zieltyp>")`.
Ersetze die Platzhalter — ein Tupel pro Spalte, die du behalten willst.

In [ ]:
# Fülle die Mapping-Tupel aus. Struktur je Zeile:
#   ("<quellspalte>", "<quelltyp>", "<zielspalte>", "<zieltyp>")
#   ____mappings : welche Rohspalte wird auf welchen Namen/Typ abgebildet?
mapped = ApplyMapping.apply(
    frame=orders,
    mappings=[
        ("<quellspalte>", "<typ>", "<zielspalte>", "<typ>"),
        # ... weitere Tupel ergänzen, eins pro Spalte
    ],
)
mapped.printSchema()

## 3) Filter — nur versendete Bestellungen

`Filter.apply` behält nur Zeilen, für die die Lambda-Funktion `True` liefert.
Achte auf **exakte** Gleichheit: ein Wert wie `"shipped, partial"` ist nicht
dasselbe wie `"shipped"`. Fülle die `____` in der Bedingung.

In [ ]:
# Fülle die ____ aus.
#   ____spalte : welche Spalte entscheidet, ob eine Bestellung versendet ist?
#   ____wert   : welcher exakte Statuswert bedeutet "versendet"?
shipped = Filter.apply(frame=mapped, f=lambda row: row["____spalte"] == "____wert")
print("shipped rows:", shipped.count())

## 4) toDF + Spark SQL — Aggregation

`toDF()` macht aus dem DynamicFrame einen Spark-DataFrame. Über eine temporäre
View kannst du ihn dann mit SQL abfragen. Der View-Name in `createOrReplaceTempView`
muss derselbe sein wie in der `FROM`-Klausel. Fülle die `____`.

In [ ]:
# Fülle die ____ aus. DynamicFrame -> DataFrame -> temp View -> Spark SQL.
#   ____view   : unter welchem Namen sprichst du die Tabelle in SQL an?
#   ____gruppe : nach welcher Spalte willst du gruppieren und zählen?
df = mapped.toDF()
df.createOrReplaceTempView("____view")
spark.sql(
    "SELECT ____gruppe, count(*) AS n "
    "FROM ____view GROUP BY ____gruppe ORDER BY n DESC"
).show(truncate=False)

## 5) Parquet schreiben

Schreib das Ergebnis als Parquet in einen **eigenen** Notebook-Prefix (nicht den
Job-Output aus Ü5.1 überschreiben). Denk an den Schreibmodus, damit ein zweiter
Lauf nicht scheitert. Fülle Zielpfad und `____modus`.

In [ ]:
# Fülle die ____ aus.
#   <bucket>/<pfad> : wohin sollen die Parquet-Dateien? (eigener Notebook-Prefix)
#   ____modus       : welcher Schreibmodus, damit ein Re-Run nicht an alten Dateien scheitert?
OUTPUT = "s3://<bucket>/<pfad>/"
mapped.toDF().write.mode("____modus").parquet(OUTPUT)
print("geschrieben nach", OUTPUT)

## 6) Zählen

Zum Abschluss: wie viele Zeilen stecken im gemappten Frame? Fülle den `____`.

In [ ]:
# Fülle die ____ aus.
#   ____frame : von welchem Frame willst du die Zeilen zählen?
print("Zeilen gesamt:", ____frame.count())